# Notebook A: build the LSTM training dataset

Turns monthly vegetation-index GeoTIFFs and daily historical weather into one file, `training_dataset.npz`, which Notebook B reads.

**Main arrays**

| Array | Shape | Content |
|---|---|---|
| `vi` | pixels × months × indices | monthly VI and Landsat band values (NaN if missing) |
| `qa` | pixels × months × indices | 0 = clear, 2 = gap-filled, 255 = missing |
| `wx` | pixels × months × 4 | temperature (°C), rain (mm/month), RH (%), wind (m/s) |
| `static` | pixels × 5 | elevation, slope, aspect (sin, cos), TPI |
| `county_id`, `dz` | pixels | county index, elevation offset used for the weather |
| `ebin`, `vbin`, `vi_level` | pixels | sampling strata and mean reference VI |
| `x`, `y`, `row`, `col` | pixels | location on the county grid |
| `days`, `vi_stats` | months / indices | days per month; p01, p50, p99, mean, std per index |

**How it works**
- Training uses a stratified random sample (50k pixels per county); Notebook C applies the model to every pixel.
- Training weather goes through the same chain as the forecast: coarse field → bilinear → lapse-rate correction with `dz`.
- `dz` and the RH offset are recovered from the corrected 30 m forecast files and rebuilt automatically if those files change.
- Rain is a monthly total by default (`RAIN_UNIT`).
- Extra bands are stored in `vi` too: `L_COASTAL`, `L_BLUE`, `L_LST_K` (Landsat) and `S_RED`, `S_NIR`, `S_RE3`, `S_COASTAL`, `S_WV` (Sentinel-2).

**Folders** (set in the next cell)
- Coarse forecast: `F:\VI_prediction\FORECASTED_WEATHER_BC_V2`
- Corrected 30 m forecast: `F:\VI_prediction\FORECASTED_WEATHER_30M_BC_V2`
- All outputs: `F:\new_predict\VI_LSTM`

**Run order:** settings → helpers → sections 2 to 6 → checks (7 and 8).

In [ ]:
import os, re, glob, json, calendar, time, hashlib, warnings
import numpy as np, pandas as pd
import rasterio, rasterio.crs
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform as warp_transform
from rasterio.windows import Window
from scipy import ndimage
from concurrent.futures import ThreadPoolExecutor
import matplotlib.pyplot as plt

# ---------------- SETTINGS ----------------
COUNTIES = ["Kakamega", "Kilifi", "Nyeri"]

# Monthly VI files; indices not on disk yet are skipped
VI_ROOT = r"F:\VI_prediction"
VI_SUBDIR = {"SAVI": os.path.join("SAVI", "tiff"), "MTVI2": os.path.join("MTVI2", "tiff")}

# Extra inputs from multi-band files. EXTRA = name: (file pattern, value band, QA band or None). One name = one input.
# Landsat monthly stack: no QA band
LANDSAT_FILE = os.path.join(VI_ROOT, "LANDSAT_STACK", "{county}", "LANDSAT_STACK_{county}_{year}_{month:02d}.tif")
# Sentinel-2 stack 1: band 3 = B4 (red), 6 = B7 (red-edge 3), 7 = B8 (NIR), 11 = fill_flag
S2_FILE  = os.path.join(VI_ROOT, "Sentinel_AllBands_2024", "{county}", "S2_{county}_{year}_{month:02d}.tif")
# Sentinel-2 stack 2: band 1 = B1 (coastal), 2 = B9 (water vapour), 3 = fill_flag
CWV_DIR  = {"Kakamega": "C_WV_KACH", "Kilifi": "C_WV_KILIFI", "Nyeri": "C_WV_NYERI"}
CWV_FILE = os.path.join(VI_ROOT, "coastal_water_vapour", "{cwv_dir}", "S2_CWV_{county}_{year}_{month:02d}.tif")
EXTRA = {
    "L_COASTAL": (LANDSAT_FILE, 1, None), "L_BLUE": (LANDSAT_FILE, 2, None), "L_LST_K": (LANDSAT_FILE, 3, None),
    "S_RED": (S2_FILE, 3, 11), "S_NIR": (S2_FILE, 7, 11), "S_RE3": (S2_FILE, 6, 11),
    "S_COASTAL": (CWV_FILE, 1, 3), "S_WV": (CWV_FILE, 2, 3),
}

VIS = ["MCARI", "NDMI", "NDRE", "NDVI", "PRI", "PSRI", "RDVI", "CVI", "MTVI2", "SAVI"] + list(EXTRA)
START_MONTH, END_MONTH = "2020-01", "2026-09"   # last complete month
NODATA = -9999.0

# Historical daily weather (file patterns)
HIST_ROOT = r"F:\VI_prediction\HISTORICAL WEATHER"
# Temperature = (tmax + tmin) / 2
HIST_WX = {
    "rain": os.path.join(HIST_ROOT, "CHIRPS_DATA_PY_daily", "filled_daily", "{county}", "CHIRPS_{county}_{year}_{month:02d}_*.tif"),
    "temp_max": os.path.join(HIST_ROOT, "ERA5_LAND_TEMP_fixed", "{county}", "tmax", "{county}_tmax_{year}{month:02d}*.tif"),
    "temp_min": os.path.join(HIST_ROOT, "ERA5_LAND_TEMP_fixed", "{county}", "tmin", "{county}_tmin_{year}{month:02d}*.tif"),
    "rh":   os.path.join(HIST_ROOT, "DAILY_RH", "RH_{county_upper}", "daily_tifs", "{county}_RH_{year}-{month:02d}-*.tif"),
    "wind": os.path.join(HIST_ROOT, "WIND_DATA_PY_daily", "county_daily_direct", "{county}", "WIND_{county}_{year}_{month:02d}_*.tif"),
}
WIND_BAND = 1               # band 1 = wind speed
MIN_DAY_FRAC = 0.8          # min share of daily files per month

# Forecast folders (used to recover dz and the RH offset)
IN_DIR   = r"F:\VI_prediction\FORECASTED_WEATHER_BC_V2"       # coarse forecast (read by the correction notebook)
FC30_DIR = r"F:\VI_prediction\FORECASTED_WEATHER_30M_BC_V2"   # corrected 30 m forecast (its output)
DEM_PATH = r"F:\VI_prediction\SRTM\{county}\{county_lower}_dem_clipped.tif"
LAPSE_T, LAPSE_TD = -6.5 / 1000, -2.0 / 1000
DEM_SMOOTH_PX, TPI_WIN_PX = 3, 15

# Sampling
WORK_DIR = r"F:\new_predict\VI_LSTM"      # all outputs of this notebook go here
SAMPLES_PER_COUNTY = 50000   # final pixels kept per county (50k-100k is typical)
CAND_FACTOR   = 2.0          # candidates drawn = this x samples
ELEV_BINS     = 10           # elevation strata (deciles of the county's elevation)
VI_BINS       = 4            # VI-level strata per county
REF_VI        = "NDVI"       # index that defines VI level
STRATA_MIX    = 0.5          # 1 = equal per stratum, 0 = proportional
MIN_USABLE_FRAC = 0.5        # min share of usable values per pixel
SEED, READ_WORKERS = 42, 4
RAIN_UNIT = "mm_month"        # "mm_month" or "mm_day"
WX_VERSION = "v2"              # change to force a weather rebuild
N_RH_DATES = 6                # dates used for the RH offset
USE_RH_OFFSET = True          # apply the recovered RH offset
WXN = ["temp", "rain", "rh", "wind"]
STATIC_NAMES = ["elev", "slope", "aspect_sin", "aspect_cos", "tpi"]

STATIC_DIR = os.path.join(WORK_DIR, "_static"); os.makedirs(STATIC_DIR, exist_ok=True)
CACHE_DIR  = os.path.join(WORK_DIR, "_cache");  os.makedirs(CACHE_DIR, exist_ok=True)
static_path = lambda c: os.path.join(STATIC_DIR, f"{c}_static.tif")
dz_path     = lambda c: os.path.join(STATIC_DIR, f"{c}_dz.tif")

def month_list(a, b):
    y, m = map(int, a.split("-")); y2, m2 = map(int, b.split("-")); out = []
    while (y, m) <= (y2, m2):
        out.append((y, m)); y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out
MONTHS = month_list(START_MONTH, END_MONTH)
MLAB = [f"{y}-{m:02d}" for y, m in MONTHS]
def vi_file(idx, c, y, m):
    if idx in EXTRA:
        return EXTRA[idx][0].format(county=c, year=y, month=m, cwv_dir=CWV_DIR.get(c, c))
    return os.path.join(VI_ROOT, VI_SUBDIR.get(idx, idx), c, f"{idx}_{c}_{y}_{m:02d}.tif")

def band_of(idx):
    e = EXTRA.get(idx)
    return (e[1], e[2]) if e else (None, None)

VIS_USED = [v for v in VIS if any(os.path.exists(vi_file(v, c, y, m)) for c in COUNTIES for (y, m) in MONTHS[:12])]
print("indices found on disk :", VIS_USED)
print("indices NOT found yet :", [v for v in VIS if v not in VIS_USED])
print(f"{len(MONTHS)} months: {MLAB[0]} .. {MLAB[-1]}")

## 1. Helpers
Humidity formulas, resampling and the historical-weather chain. Notebook C uses the same code.

In [ ]:
A_MAGNUS, B_MAGNUS = 17.625, 243.04
def rh_from_td(t, td):
    return np.clip(100 * np.exp(A_MAGNUS * td / (B_MAGNUS + td)) / np.exp(A_MAGNUS * t / (B_MAGNUS + t)), 0, 100)
def td_from_rh(t, rh):
    g = np.log(np.clip(rh, 1, 100) / 100) + A_MAGNUS * t / (B_MAGNUS + t)
    return B_MAGNUS * g / (A_MAGNUS - g)

def fill_nearest(a):
    """Fill NaN with the nearest valid value."""
    m = np.isnan(a)
    if not m.any() or m.all():
        return a
    idx = ndimage.distance_transform_edt(m, return_distances=False, return_indices=True)
    return a[tuple(idx)]

def to_crs(xs, ys, src_crs, dst_crs):
    if rasterio.crs.CRS.from_user_input(src_crs) == rasterio.crs.CRS.from_user_input(dst_crs):
        return np.asarray(xs), np.asarray(ys)
    x2, y2 = warp_transform(src_crs, dst_crs, np.asarray(xs).tolist(), np.asarray(ys).tolist())
    return np.asarray(x2), np.asarray(y2)

class Pts:
    """A set of points (xs, ys in `crs`); coordinates in other CRSs are computed once and."""
    def __init__(self, xs, ys, crs):
        self.xs, self.ys, self.crs, self._c = np.asarray(xs), np.asarray(ys), crs, {}
    def in_crs(self, crs):
        k = str(crs)
        if k not in self._c:
            self._c[k] = to_crs(self.xs, self.ys, self.crs, crs)
        return self._c[k]

def sample_bilinear(arr, tf, xs, ys):
    """Bilinear sample of a NaN-free array at points in the raster's CRS."""
    cf, rf = (~tf) * (xs, ys)
    c, r = cf - 0.5, rf - 0.5
    h, w = arr.shape
    c, r = np.clip(c, 0, w - 1), np.clip(r, 0, h - 1)
    c0, r0 = np.floor(c).astype(int), np.floor(r).astype(int)
    c1, r1 = np.minimum(c0 + 1, w - 1), np.minimum(r0 + 1, h - 1)
    fc, fr = c - c0, r - r0
    top = arr[r0, c0] * (1 - fc) + arr[r0, c1] * fc
    bot = arr[r1, c0] * (1 - fc) + arr[r1, c1] * fc
    return (top * (1 - fr) + bot * fr).astype("float32")

def sample_tif_nearest(path, pts, band=1):
    """Nearest-pixel values of a GeoTIFF at the points (NaN outside the file)."""
    with rasterio.open(path) as s:
        x2, y2 = pts.in_crs(s.crs)
        cols, rows = (~s.transform) * (x2, y2)
        cols, rows = np.floor(cols).astype(int), np.floor(rows).astype(int)
        inside = (rows >= 0) & (rows < s.height) & (cols >= 0) & (cols < s.width)
        out = np.full(len(x2), np.nan, "float32")
        if not inside.any():
            return out
        r0, r1, c0, c1 = rows[inside].min(), rows[inside].max() + 1, cols[inside].min(), cols[inside].max() + 1
        a = s.read(band, window=Window(c0, r0, c1 - c0, r1 - r0), masked=True).astype("float32").filled(np.nan)
        out[inside] = a[rows[inside] - r0, cols[inside] - c0]
    return out

def _date_of(path):
    for mt in re.finditer(r"(?<!\d)(\d{4})[-_]?(\d{2})[-_]?(\d{2})(?!\d)", os.path.basename(path)):
        y, mo, d = map(int, mt.groups())
        if 2000 <= y <= 2100 and 1 <= mo <= 12 and 1 <= d <= 31:
            return y, mo, d
    return None

def daily_files(var, county, y, m):
    tpl = HIST_WX[var]
    if "<EDIT" in tpl:
        raise ValueError(f"HIST_WX['{var}'] still contains a placeholder: edit it in the settings cell")
    files = sorted(f for f in glob.glob(tpl.format(county=county, county_lower=county.lower(), county_upper=county.upper(), year=y, month=m))
                   if f.lower().endswith(".tif"))
    out = {}
    for f in files:
        d = _date_of(f)
        if d and d[0] == y and d[1] == m:
            out[d[2]] = f
    if not out and files:                      # no date in the file names: assume one file per day in order
        out = {i + 1: f for i, f in enumerate(files[:calendar.monthrange(y, m)[1]])}
    return out

def fix_units(var, a):
    if var.startswith("temp") and np.nanmean(a) > 100:      # Kelvin -> degC
        a = a - 273.15
    if var == "rh" and np.nanmax(a) <= 1.5:        # fraction -> %
        a = a * 100
    if var == "rain":
        a = np.where(a < 0, np.nan, a)             # CHIRPS no-data (-9999)
    return a

def wind_band_of(src):
    """Band number holding wind speed."""
    if WIND_BAND is not None:
        return WIND_BAND
    if src.count == 1:
        return 1
    for i, d in enumerate(src.descriptions, 1):
        if d and "speed" in d.lower():
            return i
    raise ValueError(f"cannot tell which band is wind speed in {src.name}: band descriptions = {src.descriptions}. Set WIND_BAND in the settings cell.")

def _monthly_single(key, county, y, m):
    """Monthly field from the daily files of one variable (rain: total or mean, see RAIN_UNIT)."""
    var = "temp" if key.startswith("temp") else key
    files = daily_files(key, county, y, m)
    nd = calendar.monthrange(y, m)[1]
    if len(files) < MIN_DAY_FRAC * nd:
        return None
    acc = cnt = None
    for d, f in files.items():
        with rasterio.open(f) as s:
            b = wind_band_of(s) if var == "wind" else 1
            a = s.read(b, masked=True).astype("float32").filled(np.nan); tf, crs = s.transform, s.crs
        a = fix_units(var, a)
        ok = np.isfinite(a)
        acc = np.where(ok, a, 0) if acc is None else acc + np.where(ok, a, 0)
        cnt = ok.astype("float32") if cnt is None else cnt + ok
    mean = np.where(cnt > 0, acc / np.maximum(cnt, 1), np.nan).astype("float32")
    arr = mean * nd if (var == "rain" and RAIN_UNIT == "mm_month") else mean
    return fill_nearest(arr), tf, crs

def monthly_coarse(var, county, y, m):
    """Temperature = (monthly tmax + tmin) / 2; other variables are read directly."""
    if var != "temp":
        return _monthly_single(var, county, y, m)
    hi, lo = _monthly_single("temp_max", county, y, m), _monthly_single("temp_min", county, y, m)
    if hi is None or lo is None:
        return None
    if hi[0].shape != lo[0].shape or hi[1] != lo[1]:
        raise ValueError(f"tmax and tmin grids differ for {county} {y}-{m:02d}")
    return ((hi[0] + lo[0]) / 2).astype("float32"), hi[1], hi[2]

def wx_month_at(pts, dz, county, y, m, cache, rh_off=0.0):
    """Monthly [temp, rain, rh, wind] at the points: bilinear, then lapse correction with dz."""
    out = np.full((len(pts.xs), 4), np.nan, "float32")
    got = {}
    for v in ("temp", "rain", "rh", "wind"):
        key = (v, county, y, m)
        if key not in cache:
            cache[key] = monthly_coarse(v, county, y, m)
        if cache[key] is None:
            return out
        arr, tf, crs = cache[key]
        x, yy = pts.in_crs(crs)
        got[v] = sample_bilinear(arr, tf, x, yy)
    T, RH = got["temp"], got["rh"]
    Td = td_from_rh(T, RH)
    Tc, Tdc = T + LAPSE_T * dz, Td + LAPSE_TD * dz
    rh = np.clip(rh_from_td(Tc, Tdc) + rh_off, 0, 100)
    out[:, 0], out[:, 1], out[:, 2], out[:, 3] = Tc, got["rain"], rh, got["wind"]
    return out

def pts_hash(pts):
    """Short fingerprint of the sample points, used in cache names."""
    return hashlib.md5(np.round(pts.xs, 2).tobytes() + np.round(pts.ys, 2).tobytes()).hexdigest()[:10]


## 2. Static layers and elevation offset `dz`
- **Static layers** (elevation, slope, aspect, TPI) are computed from the DEM on the VI grid.
- **`dz`** = (corrected 30 m temperature − interpolated coarse forecast) ÷ lapse rate, averaged over 3 dates. Any re-centring done in the correction notebook is included automatically.
- **RH offset** = mean gap between the corrected 30 m RH and the plain chain (about 0 if no RH anchoring was applied).
- If the corrected files change, `dz`, the RH offset and the weather cache are rebuilt on the next run.

In [ ]:
def nan_smooth(a, size, keep_mask=True):
    if size <= 1: return a
    m = np.isfinite(a)
    s = ndimage.uniform_filter(np.where(m, a, 0).astype("float64"), size, mode="nearest")
    c = ndimage.uniform_filter(m.astype("float64"), size, mode="nearest")
    with np.errstate(invalid="ignore", divide="ignore"):
        out = s / c
    out[c < 1e-6] = np.nan
    if keep_mask: out[~m] = np.nan
    return out.astype("float32")

def vi_template(county, vis):
    """Reference grid (projected CRS) for the county."""
    first = None
    for k in vis:
        for (y, m) in MONTHS:
            p = vi_file(k, county, y, m)
            if os.path.exists(p):
                with rasterio.open(p) as s:
                    d = dict(path=p, crs=s.crs, transform=s.transform, shape=(s.height, s.width))
                if d["crs"] is not None and d["crs"].is_projected:
                    return d
                first = first or d
                break
    if first is None:
        raise FileNotFoundError(f"no VI file found for {county}")
    raise ValueError(f"{county}: no VI file with a projected CRS found (first file {first['path']} is {first['crs']}); "
                     "slope/TPI need metres. Put an index that is exported in UTM (e.g. MCARI) first in VIS.")

def build_static(county, tpl):
    path = static_path(county)
    if os.path.exists(path): return path
    h, w = tpl["shape"]
    elev = np.full((h, w), np.nan, "float32")
    with rasterio.open(DEM_PATH.format(county=county, county_lower=county.lower())) as src:
        reproject(rasterio.band(src, 1), elev, dst_transform=tpl["transform"], dst_crs=tpl["crs"],
                  resampling=Resampling.average, src_nodata=src.nodata if src.nodata is not None else -9999, dst_nodata=np.nan)
    elev = nan_smooth(elev, DEM_SMOOTH_PX)
    inside = np.isfinite(elev)
    fill = fill_nearest(elev)
    px = abs(tpl["transform"].a)
    gy, gx = np.gradient(fill, px)               # gy: change per metre going south (row direction)
    gn = -gy                                      # change per metre going north
    norm = np.hypot(gx, gn)
    slope = np.degrees(np.arctan(norm))
    flat = slope < 0.5
    asin = np.where(flat, 0, -gx / np.maximum(norm, 1e-9))     # downslope direction (clockwise from north): sin, cos
    acos = np.where(flat, 0, -gn / np.maximum(norm, 1e-9))
    tpi = fill - nan_smooth(fill, TPI_WIN_PX, keep_mask=False)
    stack = np.stack([elev, slope, asin, acos, tpi]).astype("float32")
    stack[:, ~inside] = np.nan
    prof = dict(driver="GTiff", height=h, width=w, count=5, dtype="float32", crs=tpl["crs"], transform=tpl["transform"],
                nodata=np.nan, compress="deflate", tiled=True, predictor=3)
    with rasterio.open(path, "w", **prof) as d:
        d.write(stack)
        for i, n in enumerate(STATIC_NAMES, 1): d.set_band_description(i, n)
    print(f"  {county}: static layers written ({np.nanmin(elev):.0f}-{np.nanmax(elev):.0f} m)")
    return path

rhoff_path = lambda c: os.path.join(STATIC_DIR, f"{c}_rhoff.json")

def _fc30_files(county, group, name):
    return sorted(f for f in glob.glob(os.path.join(FC30_DIR, group, name, "mean", f"{county}_{name}_mean_*.tif"))
                  if re.search(r"_\d{4}-\d{2}-\d{2}\.tif$", f))

def fc_signature(county):
    """Fingerprint of the corrected forecast files."""
    parts = []
    for g, v in (("temperature", "temp"), ("humidity", "rh")):
        fs = _fc30_files(county, g, v); parts.append(f"{v}:{len(fs)}")
        for i in sorted({0, len(fs) // 2, len(fs) - 1}) if fs else []:
            st = os.stat(fs[i]); parts.append(f"{os.path.basename(fs[i])}:{st.st_size}:{int(st.st_mtime)}")
    return hashlib.md5("|".join(parts).encode()).hexdigest()

def _coarse_on_grid(group, name, county, d, tf, crs, shape):
    with rasterio.open(os.path.join(IN_DIR, group, name, "mean", f"{county}_{name}_mean_{d}.tif")) as s:
        a = s.read(1, masked=True).astype("float32").filled(np.nan); ctf, ccrs = s.transform, s.crs
    out = np.full(shape, np.nan, "float32")
    reproject(fill_nearest(a), out, src_transform=ctf, src_crs=ccrs, dst_transform=tf, dst_crs=crs,
              resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    return out

def build_dz(county):
    """Recover dz and the RH offset from the corrected 30 m files."""
    path, sigp, rhp = dz_path(county), dz_path(county) + ".sig", rhoff_path(county)
    sig = fc_signature(county)
    if all(os.path.exists(p) for p in (path, sigp, rhp)) and open(sigp).read().strip() == sig:
        return path
    if os.path.exists(path): print(f"  {county}: corrected forecast files changed since dz was built -> rebuilding dz, RH offset and (via the cache name) weather")
    files = _fc30_files(county, "temperature", "temp")
    if not files: raise FileNotFoundError(f"no corrected 30 m temperature files for {county} in {FC30_DIR}")
    dzs = []
    for i in sorted({0, len(files) // 2, len(files) - 1}):
        d = re.search(r"_(\d{4}-\d{2}-\d{2})\.tif$", files[i]).group(1)
        with rasterio.open(files[i]) as s:
            T30 = s.read(1, masked=True).astype("float32").filled(np.nan); tf, crs, shape, prof = s.transform, s.crs, s.shape, s.profile
        Tb = _coarse_on_grid("temperature", "temp", county, d, tf, crs, shape)
        dzs.append(((T30 - Tb) / LAPSE_T).astype("float32"))
        del T30, Tb
    dz = np.nanmean(np.stack(dzs), 0).astype("float32")
    spread = np.nanmedian(np.abs(dzs[0] - dzs[-1])[::7, ::7]) if len(dzs) > 1 else float("nan")
    prof.update(count=1, dtype="float32", nodata=np.nan, compress="deflate", predictor=3, tiled=True, blockxsize=256, blockysize=256)
    with rasterio.open(path, "w", **prof) as d_:
        d_.write(dz, 1)
    print(f"  {county}: dz written | range {np.nanmin(dz):.0f}..{np.nanmax(dz):.0f} m | mean {np.nanmean(dz):+.0f} m | "
          f"median difference between dates {spread:.2f} m (should be ~0)")
    rfiles = _fc30_files(county, "humidity", "rh")
    offs = []
    for i in sorted({int(round(x)) for x in np.linspace(0, len(rfiles) - 1, N_RH_DATES)}) if rfiles else []:
        d = re.search(r"_(\d{4}-\d{2}-\d{2})\.tif$", rfiles[i]).group(1)
        fT = os.path.join(FC30_DIR, "temperature", "temp", "mean", f"{county}_temp_mean_{d}.tif")
        if not os.path.exists(fT) or not os.path.exists(os.path.join(IN_DIR, "humidity", "rh", "mean", f"{county}_rh_mean_{d}.tif")): continue
        with rasterio.open(rfiles[i]) as s: RH30 = s.read(1, masked=True).astype("float32").filled(np.nan)
        Tb  = _coarse_on_grid("temperature", "temp", county, d, tf, crs, shape)
        RHb = _coarse_on_grid("humidity", "rh", county, d, tf, crs, shape)
        Tc, Tdc = Tb + LAPSE_T * dz, td_from_rh(Tb, RHb) + LAPSE_TD * dz
        offs.append(float(np.nanmean((RH30 - rh_from_td(Tc, Tdc))[::5, ::5])))
        del RH30, Tb, RHb, Tc, Tdc
    rh_off = float(np.median(offs)) if offs else 0.0
    json.dump(dict(rh_offset=rh_off, per_date=offs, n_dates=len(offs)), open(rhp, "w"))
    print(f"  {county}: RH offset (corrected 30 m RH - plain chain) = {rh_off:+.2f} % | per date {np.round(offs, 2).tolist()}")
    open(sigp, "w").write(sig)
    return path

TPL = {}
for c in COUNTIES:
    TPL[c] = vi_template(c, VIS_USED)
    print(c, "VI grid:", TPL[c]["shape"], TPL[c]["crs"])
    build_static(c, TPL[c]); build_dz(c)

# values used by the weather section
RH_OFF = {c: (json.load(open(rhoff_path(c)))["rh_offset"] if USE_RH_OFFSET else 0.0) for c in COUNTIES}
FC_SIG = {c: open(dz_path(c) + ".sig").read().strip()[:8] for c in COUNTIES}
print("RH offset applied to the training chain (%):", {c: round(v, 2) for c, v in RH_OFF.items()})


## 3. Candidate pixels
Random pixels, equal numbers per elevation decile. Extra candidates are drawn because some are dropped later.

In [ ]:
def draw_points(county, tpl, n_cand):
    with rasterio.open(static_path(county)) as s:
        elev = s.read(1)
    rng = np.random.default_rng(SEED + COUNTIES.index(county))
    flat_valid = np.flatnonzero(np.isfinite(elev).ravel())
    vals = elev.ravel()[flat_valid]
    sub = vals[rng.integers(0, len(vals), min(2_000_000, len(vals)))]
    edges = np.quantile(sub, np.linspace(0, 1, ELEV_BINS + 1)[1:-1])
    b = np.digitize(vals, edges)
    per = n_cand // ELEV_BINS
    flat, ebin = [], []
    for k in range(ELEV_BINS):
        pick = rng.choice(flat_valid[b == k], size=min(per, int((b == k).sum())), replace=False)
        flat.append(pick); ebin.append(np.full(len(pick), k, "int8"))
    flat, ebin = np.concatenate(flat), np.concatenate(ebin)
    rows, cols = np.divmod(flat, elev.shape[1])
    xs, ys = tpl["transform"] * (cols + 0.5, rows + 0.5)
    return rows, cols, np.asarray(xs), np.asarray(ys), ebin

POINTS, PTS = {}, {}
for c in COUNTIES:
    r, cl, x, y, eb = draw_points(c, TPL[c], int(SAMPLES_PER_COUNTY * CAND_FACTOR))
    POINTS[c] = dict(rows=r, cols=cl, xs=x, ys=y, ebin=eb)
    PTS[c] = Pts(x, y, TPL[c]["crs"])
    print(f"{c}: {len(r)} candidate pixels (sample fingerprint {pts_hash(PTS[c])})")

## 4. Monthly VI and QA at the sample pixels
- Files are sampled by coordinates, so different grids and CRSs can be mixed.
- Band 2 is used as QA when present; otherwise a valid value counts as QA 0.
- Landsat files have no QA band (valid value = QA 0). Sentinel-2 files use their `fill_flag` band as QA.
- Missing files stay NaN with QA 255.
- Results are cached per county and index. If you add or replace files for a cached index, delete its file in `_cache`.

In [ ]:
def sample_vi_file(path, pts, band=None, qa_band=None):
    """Value and QA at the points, found by coordinates."""
    n = len(pts.xs)
    v = np.full(n, np.nan, "float32"); q = np.full(n, 255, "float32")
    with rasterio.open(path) as s:
        x2, y2 = pts.in_crs(s.crs)
        cc, rr = (~s.transform) * (x2, y2)
        cc, rr = np.floor(cc).astype(int), np.floor(rr).astype(int)
        inside = (rr >= 0) & (rr < s.height) & (cc >= 0) & (cc < s.width)
        if not inside.any():
            return v, q
        r0, r1, c0, c1 = rr[inside].min(), rr[inside].max() + 1, cc[inside].min(), cc[inside].max() + 1
        win = Window(c0, r0, c1 - c0, r1 - r0)
        if band:
            bands = [band, qa_band] if qa_band else [band]
        else:
            bands = [1, 2] if s.count >= 2 else [1]
        d = s.read(bands, window=win).astype("float32")           # raw values (nodata is not masked here)
    ri, ci = rr[inside] - r0, cc[inside] - c0
    v[inside] = d[0][ri, ci]
    if len(bands) == 2:
        q[inside] = d[1][ri, ci]
    else:
        q[inside] = np.where(np.isfinite(v[inside]) & (v[inside] != NODATA), 0, 255)
    return v, q

def read_vi_index(county, idx, pts):
    """One index, all months: values, QA, files found, bands."""
    P, M = len(pts.xs), len(MONTHS)
    vi = np.full((P, M), np.nan, "float32"); qa = np.full((P, M), 255, "uint8")
    have = [mi for mi, (y, m) in enumerate(MONTHS) if os.path.exists(vi_file(idx, county, y, m))]
    if not have:
        return vi, qa, 0, 0
    for mi in have:                                    # first readable file: coordinate cache + band count
        try:
            with rasterio.open(vi_file(idx, county, *MONTHS[mi])) as s:
                pts.in_crs(s.crs); n_bands = (2 if EXTRA[idx][2] else 1) if idx in EXTRA else s.count
            break
        except rasterio.errors.RasterioError:
            continue
    else:
        BAD_FILES.extend(vi_file(idx, county, *MONTHS[mi]) for mi in have)
        return vi, qa, len(have), 0
    def task(mi):
        p = vi_file(idx, county, *MONTHS[mi])
        try:
            return mi, sample_vi_file(p, pts, *band_of(idx))
        except rasterio.errors.RasterioError:          # unreadable file: treated as missing
            BAD_FILES.append(p)
            return mi, (np.full(P, np.nan, "float32"), np.full(P, 255, "float32"))
    t0 = time.time()
    with ThreadPoolExecutor(READ_WORKERS) as ex:
        for n, (mi, (v, q)) in enumerate(ex.map(task, have), 1):
            if idx.startswith("S_"):
                q = np.where(q == 1, 2, q)      # ±1 month fill -> gap-filled (QA 2)
            ok = np.isfinite(v) & (v != NODATA) & np.isin(q, (0, 2))
            vi[:, mi] = np.where(ok, v, np.nan)
            qa[:, mi] = np.where(ok, q, 255)
            if n % 24 == 0: print(f"    {county} {idx}: {n}/{len(have)} files ({time.time()-t0:.0f}s)", flush=True)
    return vi, qa, len(have), n_bands

BAD_FILES = []
VI_DATA, VI_BANDS = {}, {}
for c in COUNTIES:
    h = pts_hash(PTS[c]); vl, ql = [], []
    for idx in VIS_USED:
        cache = os.path.join(CACHE_DIR, f"{c}_vi_{idx}_{h}_{MLAB[0]}_{MLAB[-1]}.npz")
        n_disk = sum(os.path.exists(vi_file(idx, c, y, m)) for (y, m) in MONTHS)
        src = None
        if os.path.exists(cache):
            z = np.load(cache)
            if int(z["n_files"]) == n_disk:
                v, q, nf, nb_ = z["vi"], z["qa"], int(z["n_files"]), int(z["n_bands"]); src = "cache"
            else:
                print(f"  {c} {idx}: cache has {int(z['n_files'])} files, disk has {n_disk} -> re-reading")
        if src is None:
            v, q, nf, nb_ = read_vi_index(c, idx, PTS[c])
            np.savez(cache, vi=v, qa=q, n_files=nf, n_bands=nb_); src = "read"
        vl.append(v); ql.append(q); VI_BANDS.setdefault(idx, {})[c] = nb_
        usable_frac = float((np.isin(q, (0, 2)) & np.isfinite(v)).mean())
        flag = "" if nf == len(MONTHS) else f"  <-- only {nf}/{len(MONTHS)} monthly files found"
        print(f"{c:9s} {idx:6s} {src:5s} files {nf:3d}/{len(MONTHS)} | bands {nb_} | usable {usable_frac:.2f}{flag}")
    VI_DATA[c] = (np.stack(vl, 2), np.stack(ql, 2))

if BAD_FILES:
    bad = sorted(set(BAD_FILES))
    open(os.path.join(WORK_DIR, "bad_vi_files.txt"), "w").write("\n".join(bad))
    print(f"\nWARNING: {len(bad)} unreadable files were treated as missing (list saved to bad_vi_files.txt):")
    print(*bad, sep="\n")

In [ ]:
# Check: fill_flag values in the Sentinel stacks (expected 0, 1, 2, 255)
for c in COUNTIES:
    for idx in ("S_RED", "S_COASTAL"):
        fs = [vi_file(idx, c, y, m) for (y, m) in MONTHS if os.path.exists(vi_file(idx, c, y, m))]
        if not fs: continue
        f = fs[len(fs) // 2]
        with rasterio.open(f) as s:
            q = s.read(EXTRA[idx][2], out_shape=(max(1, s.height // 10), max(1, s.width // 10)))
        print(c, idx, os.path.basename(f), dict(zip(*np.unique(q[np.isfinite(q)], return_counts=True))))


## 5. Historical weather at the sample pixels
- Daily files are averaged to a monthly field (rain is summed), interpolated to each pixel, then lapse-corrected with `dz`, as in the forecast.
- Months with too few daily files are NaN and retried on each run.
- Results are cached per county.

In [ ]:
from scipy.spatial import cKDTree

def fill_spatial_gaps(wx, pts):
    """Fill points with no value from the nearest sampled point that has data."""
    filled = {}
    for i, name in enumerate(WXN):
        bad = np.isnan(wx[:, :, i])                                   # points x months
        groups = {}
        for k in range(bad.shape[1]):
            if bad[:, k].any() and not bad[:, k].all():
                groups.setdefault(bad[:, k].tobytes(), []).append(k)  # months sharing the same gap pattern
        for ks in groups.values():
            b = bad[:, ks[0]]
            tree = cKDTree(np.c_[pts.xs[~b], pts.ys[~b]])
            _, j = tree.query(np.c_[pts.xs[b], pts.ys[b]])
            for k in ks:
                wx[b, k, i] = wx[~b, k, i][j]
            filled[name] = int(b.sum())
    return filled

def weather_points(county):
    pts = PTS[county]; P = len(pts.xs)
    dz = sample_tif_nearest(dz_path(county), pts)
    cache_f = os.path.join(CACHE_DIR, f"{county}_wx_{WX_VERSION}_{RAIN_UNIT}_{FC_SIG[county]}_{pts_hash(pts)}_{MLAB[0]}_{MLAB[-1]}.npz")
    wx = np.load(cache_f)["wx"] if os.path.exists(cache_f) else np.full((P, len(MONTHS), 4), np.nan, "float32")
    todo = [mi for mi in range(len(MONTHS)) if any(np.isnan(wx[:, mi, k]).all() for k in range(4))]
    if todo:
        for key in HIST_WX:                                           # fill the coordinate caches before the threads start
            for (y, m) in MONTHS:
                fs = daily_files(key, county, y, m)
                if fs:
                    with rasterio.open(next(iter(fs.values()))) as s: pts.in_crs(s.crs)
                    break
        def task(mi):
            y, m = MONTHS[mi]
            return mi, wx_month_at(pts, dz, county, y, m, {}, RH_OFF[county])
        t0 = time.time()
        with ThreadPoolExecutor(READ_WORKERS) as ex:
            for n, (mi, arr) in enumerate(ex.map(task, todo), 1):
                wx[:, mi, :] = arr
                if n % 12 == 0: print(f"  {county}: weather {n}/{len(todo)} months ({time.time()-t0:.0f}s)", flush=True)
        np.savez(cache_f, wx=wx)                                      # raw values are cached (before gap filling)
    # report months that are empty, per variable
    for k, name in enumerate(WXN):
        miss = [MLAB[mi] for mi in range(len(MONTHS)) if np.isnan(wx[:, mi, k]).all()]
        if miss: print(f"  WARNING {county}: no {name} for months {miss}")
    filled = fill_spatial_gaps(wx, pts)
    left = {n: round(float(np.isnan(wx[:, :, k]).mean()), 4) for k, n in enumerate(WXN)}
    print(f"  {county}: recomputed {len(todo)} months | points gap-filled {filled or 'none'} | NaN share left {left}")
    return wx, dz

WX_DATA, DZ_PTS = {}, {}
for c in COUNTIES:
    print(c, "building weather ...")
    WX_DATA[c], DZ_PTS[c] = weather_points(c)

## 6. Filter, stratified selection, save
1. Drop pixels with too few usable values, or without static layers or `dz`.
2. Give each pixel a VI level (quartile of its mean reference VI).
3. Pick the final sample stratified by elevation × VI level (`STRATA_MIX` balances equal vs. proportional allocation).
4. Save `training_dataset.npz` and `training_dataset_meta.json`.

In [ ]:
def allocate(sizes, n, mix):
    """Number of pixels to draw from each stratum."""
    sizes = np.asarray(sizes, float); S_ = len(sizes)
    n = int(min(n, sizes.sum()))
    w = mix / S_ + (1 - mix) * sizes / sizes.sum()
    alloc, free = np.zeros(S_), sizes > 0
    for _ in range(100):
        remaining = n - alloc.sum()
        if remaining < 1e-6 or not free.any(): break
        add = remaining * (w * free) / (w * free).sum()
        alloc = np.minimum(alloc + add, sizes)
        free = alloc < sizes - 1e-9
    a = np.floor(alloc + 1e-9).astype(int)
    left = n - int(a.sum())
    for k in np.argsort(-(alloc - a)):
        if left <= 0: break
        if a[k] < sizes[k]: a[k] += 1; left -= 1
    return a

REF = REF_VI or next((v for v in ["NDVI", "SAVI", "NDRE", "NDMI"] if v in VIS_USED), VIS_USED[0])
k_ref = VIS_USED.index(REF)
K_CORE = [k for k, v in enumerate(VIS_USED) if v not in EXTRA]   # pixel filter uses the original VIs only
print(f"VI level = mean of {REF} over usable months | {ELEV_BINS} elevation x {VI_BINS} VI strata | STRATA_MIX {STRATA_MIX} (1 = equal, 0 = proportional)\n")

rng = np.random.default_rng(SEED)
parts = {k: [] for k in ["vi", "qa", "wx", "static", "dz", "county_id", "x", "y", "row", "col", "ebin", "vbin", "vi_level"]}
for ci, c in enumerate(COUNTIES):
    vi, qa = VI_DATA[c]; wx = WX_DATA[c]; p = POINTS[c]
    with rasterio.open(static_path(c)) as s:
        st = np.stack([s.read(b)[p["rows"], p["cols"]] for b in range(1, 6)], 1)
    usable = np.isin(qa, (0, 2)) & np.isfinite(vi)
    cover = usable[:, :, K_CORE].mean(axis=(1, 2))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        level = np.nanmean(np.where(usable[:, :, k_ref], vi[:, :, k_ref], np.nan), axis=1)
    keep = (cover >= MIN_USABLE_FRAC) & np.isfinite(st).all(1) & np.isfinite(DZ_PTS[c]) & np.isfinite(level)
    cand = np.flatnonzero(keep)
    if len(cand) == 0: raise RuntimeError(f"{c}: no candidate passes the quality filter")
    vedges = np.quantile(level[cand], np.linspace(0, 1, VI_BINS + 1)[1:-1])
    vb = np.digitize(level, vedges).astype("int8")
    strat = p["ebin"].astype(int) * VI_BINS + vb
    sizes = np.bincount(strat[cand], minlength=ELEV_BINS * VI_BINS)
    alloc = allocate(sizes, SAMPLES_PER_COUNTY, STRATA_MIX)
    idx = rng.permutation(np.concatenate([rng.choice(cand[strat[cand] == k], size=a, replace=False) for k, a in enumerate(alloc) if a > 0]))
    print(f"{c}: {len(p['rows'])} candidates -> {len(cand)} pass the quality filter -> {len(idx)} kept "
          f"(VI-level class edges {np.round(vedges, 3)})")
    if len(idx) < SAMPLES_PER_COUNTY: print(f"   note: fewer than SAMPLES_PER_COUNTY available; raise CAND_FACTOR for more")
    for k, a in [("vi", vi), ("qa", qa), ("wx", wx), ("static", st)]: parts[k].append(a[idx])
    parts["dz"].append(DZ_PTS[c][idx].astype("float32"))
    parts["county_id"].append(np.full(len(idx), ci, "int8"))
    for k, a in [("x", p["xs"]), ("y", p["ys"]), ("row", p["rows"]), ("col", p["cols"]), ("ebin", p["ebin"])]: parts[k].append(a[idx])
    parts["vbin"].append(vb[idx]); parts["vi_level"].append(level[idx].astype("float32"))

data = {k: np.concatenate(v) for k, v in parts.items()}

use_all = np.isin(data["qa"], (0, 2)) & np.isfinite(data["vi"])
STAT_NAMES = ["p01", "p50", "p99", "mean", "std"]
vi_stats = np.full((len(VIS_USED), 5), np.nan, "float32")
for k in range(len(VIS_USED)):
    xk = data["vi"][:, :, k][use_all[:, :, k]]
    if xk.size: vi_stats[k] = [*np.percentile(xk, [1, 50, 99]), xk.mean(), xk.std()]

out = os.path.join(WORK_DIR, "training_dataset.npz")
np.savez_compressed(out, **data, months=np.array(MLAB), vis=np.array(VIS_USED), counties=np.array(COUNTIES),
                    crs=np.array([str(TPL[c]["crs"]) for c in COUNTIES]), wx_names=np.array(WXN), wx_units=np.array(["degC", "mm/month" if RAIN_UNIT == "mm_month" else "mm/day", "%", "m/s"]),
                    days=np.array([calendar.monthrange(y, m)[1] for y, m in MONTHS]),
                    static_names=np.array(STATIC_NAMES), vi_stats=vi_stats, vi_stat_names=np.array(STAT_NAMES), ref_vi=np.array(REF))
meta = dict(created=time.strftime("%Y-%m-%d %H:%M"), n_pixels=int(data["vi"].shape[0]),
            pixels_per_county={c: int((data["county_id"] == i).sum()) for i, c in enumerate(COUNTIES)},
            months=[MLAB[0], MLAB[-1]], n_months=len(MLAB), indices=VIS_USED, ref_vi=REF,
            qa_source={i: ("QA band" if all(b >= 2 for b in VI_BANDS[i].values()) else "no QA band: valid value = QA 0") for i in VIS_USED},
            sampling=dict(seed=SEED, elev_bins=ELEV_BINS, vi_bins=VI_BINS, strata_mix=STRATA_MIX, cand_factor=CAND_FACTOR,
                          min_usable_frac=MIN_USABLE_FRAC, samples_per_county=SAMPLES_PER_COUNTY),
            weather=dict(temp="(monthly mean tmax + monthly mean tmin) / 2, lapse-corrected with dz", rain=("monthly total (sum of daily mm) = mm/month" if RAIN_UNIT == "mm_month" else "monthly mean of daily mm = mm/day"),
                         rh=f"recomputed from dewpoint after lapse correction + RH offset from the corrected forecast {RH_OFF}", wind=f"speed, band {WIND_BAND}",
                         min_day_frac=MIN_DAY_FRAC, hist_patterns=HIST_WX),
            grids={c: dict(crs=str(TPL[c]["crs"]), shape=list(TPL[c]["shape"])) for c in COUNTIES})
json.dump(meta, open(os.path.join(WORK_DIR, "training_dataset_meta.json"), "w"), indent=2, default=str)
print(f"\nsaved {out}: {data['vi'].shape[0]} pixels x {data['vi'].shape[1]} months x {data['vi'].shape[2]} indices "
      f"({os.path.getsize(out)/1e6:.0f} MB) + training_dataset_meta.json")

## 7. Checks (look at these before moving on)

In [ ]:
vi, qa, wx, cid = data["vi"], data["qa"], data["wx"], data["county_id"]
usable = np.isin(qa, (0, 2)) & np.isfinite(vi)

# (a) share of QA classes per county and index
rows = []
for ci, c in enumerate(COUNTIES):
    for k, v in enumerate(VIS_USED):
        q = qa[cid == ci][:, :, k]
        rows.append((c, v, (q == 0).mean(), (q == 2).mean(), (q == 255).mean()))
print(pd.DataFrame(rows, columns=["county", "index", "QA0_clear", "QA2_filled", "QA255_missing"]).round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 3.5))
for ci, c in enumerate(COUNTIES):
    ax.plot(usable[cid == ci].mean(axis=(0, 2)), label=c)
ax.set_xticks(range(0, len(MLAB), 6)); ax.set_xticklabels(MLAB[::6], rotation=45)
ax.set_ylabel("usable fraction (QA 0 or 2)"); ax.legend(); plt.tight_layout(); plt.show()

for ci, c in enumerate(COUNTIES):
    w = wx[cid == ci]
    ok = np.isfinite(w).all(axis=(0, 2))
    ann = np.nanmean(w[:, :, 1], axis=0)[ok]
    if RAIN_UNIT == "mm_month":
        rain_txt = "%.0f mm/month (~%.0f mm/yr)" % (ann.mean(), ann.mean() * 12)
    else:
        rain_txt = "%.2f mm/day (~%.0f mm/yr)" % (ann.mean(), ann.mean() * 365)
    print(f"{c}: weather available in {ok.sum()}/{len(MLAB)} months | mean T {np.nanmean(w[:, :, 0]):.1f} C | "
          f"mean RH {np.nanmean(w[:, :, 2]):.0f} % | mean wind {np.nanmean(w[:, :, 3]):.1f} m/s | mean rain {rain_txt}")

In [ ]:
rows = []
for k, v in enumerate(VIS_USED):
    xk = vi[:, :, k][usable[:, :, k]]
    if xk.size == 0:
        rows.append(dict(index=v, n_usable=0)); continue
    q1, q50, q99 = np.percentile(xk, [1, 50, 99])
    rows.append(dict(index=v, n_usable=xk.size, min=xk.min(), p01=q1, p50=q50, p99=q99, max=xk.max(), mean=xk.mean()))
print(pd.DataFrame(rows).round(3).to_string(index=False))

fig, axes = plt.subplots(1, len(COUNTIES), figsize=(4.2 * len(COUNTIES), 3.2), squeeze=False)
for ci, c in enumerate(COUNTIES):
    with rasterio.open(static_path(c)) as s: e = s.read(1)
    e = e[np.isfinite(e)][::37]; se = data["static"][cid == ci, 0]
    ax = axes[0, ci]
    for arr, lab in ((e, "whole county"), (se, "sample")): ax.plot(np.sort(arr), np.linspace(0, 1, len(arr)), label=lab)
    ax.set_title(c); ax.set_xlabel("elevation (m)"); ax.set_ylabel("cumulative share"); ax.legend()
plt.tight_layout(); plt.show()

# (f) where the sample pixels are
fig, axes = plt.subplots(1, len(COUNTIES), figsize=(4.2 * len(COUNTIES), 4), squeeze=False)
for ci, c in enumerate(COUNTIES):
    m = cid == ci
    sc = axes[0, ci].scatter(data["x"][m], data["y"][m], c=data["static"][m, 0], s=1, cmap="terrain")
    axes[0, ci].set_title(f"{c}: {m.sum()} pixels"); axes[0, ci].set_aspect("equal"); axes[0, ci].tick_params(labelsize=6)
    plt.colorbar(sc, ax=axes[0, ci], label="elevation (m)", shrink=0.7)
plt.tight_layout(); plt.show()

# (g) how many pixels per elevation x VI-level stratum
fig, axes = plt.subplots(1, len(COUNTIES), figsize=(3.6 * len(COUNTIES), 3.6), squeeze=False)
for ci, c in enumerate(COUNTIES):
    m = cid == ci
    H = np.zeros((ELEV_BINS, VI_BINS), int); np.add.at(H, (data["ebin"][m], data["vbin"][m]), 1)
    axes[0, ci].imshow(H, aspect="auto", cmap="viridis"); axes[0, ci].set_title(f"{c}: pixels per stratum")
    axes[0, ci].set_xlabel(f"{REF} level (low -> high)"); axes[0, ci].set_ylabel("elevation decile (low -> high)")
    for (i, j), val in np.ndenumerate(H): axes[0, ci].text(j, i, val, ha="center", va="center", color="w", fontsize=6)
plt.tight_layout(); plt.show()

In [ ]:
##Clips extreme index values to sensible limits, recomputes the stats, and saves training_dataset_clipped.npz.
import os
import numpy as np

WORK_DIR = r"F:\new_predict\VI_LSTM"
src = os.path.join(WORK_DIR, "training_dataset.npz")
dst = os.path.join(WORK_DIR, "training_dataset_clipped.npz")

HEAVY_TAILED = ["MCARI", "CVI", "PSRI", "L_COASTAL", "L_BLUE", "L_LST_K",
                "S_RED", "S_NIR", "S_RE3", "S_COASTAL", "S_WV"]   # not bounded to [-1, 1]
MARGIN = 1.0
PHYSICAL = {"CVI": (0.0, None)}             # CVI cannot be negative
BOUNDED = (-1.0, 1.0)                       # all other indices live in [-1, 1]

d = dict(np.load(src, allow_pickle=False))
vis = [str(v) for v in d["vis"]]
sn = [str(s) for s in d["vi_stat_names"]]
vi = d["vi"]                                # pixels x months x indices

rows = []
for k, v in enumerate(vis):
    x = vi[:, :, k]
    if v in HEAVY_TAILED:
        p01, p99 = d["vi_stats"][k, sn.index("p01")], d["vi_stats"][k, sn.index("p99")]
        span = p99 - p01
        lo, hi = p01 - MARGIN * span, p99 + MARGIN * span
        plo, phi = PHYSICAL.get(v, (None, None))
        if plo is not None: lo = max(lo, plo)
        if phi is not None: hi = min(hi, phi)
    else:
        lo, hi = BOUNDED
    with np.errstate(invalid="ignore"):
        n_changed = int(((x < lo) | (x > hi)).sum())
    vi[:, :, k] = np.clip(x, lo, hi)        # NaN stays NaN
    rows.append((v, round(float(lo), 3), round(float(hi), 3), n_changed))

d["vi"] = vi

use = np.isin(d["qa"], (0, 2)) & np.isfinite(vi)
for k in range(len(vis)):
    xk = vi[:, :, k][use[:, :, k]]
    if xk.size:
        d["vi_stats"][k] = [*np.percentile(xk, [1, 50, 99]), xk.mean(), xk.std()]

np.savez_compressed(dst, **d)

print(f"{'index':6s} {'lower':>9s} {'upper':>9s} {'values clipped':>15s}")
for v, lo, hi, n in rows:
    print(f"{v:6s} {lo:9} {hi:9} {n:15d}")
print("\nsaved", dst, f"({os.path.getsize(dst)/1e6:.0f} MB)")

## 8. Forecast-consistency check
Compares the training weather with the 30 m forecast files for the same calendar month. A forecast far outside the historical range (|z| > 3) usually means a unit or offset mismatch.

In [ ]:
FC_VARS = {"temp": ("temperature", "temp"), "rain": ("rainfall", "rain"), "rh": ("humidity", "rh"), "wind": ("wind", "wind_speed")}
rows = []
for ci, c in enumerate(COUNTIES):
    sel = np.flatnonzero(cid == ci)[::10]                 # every 10th point (5,000) is plenty for a county mean
    pts = Pts(data["x"][sel], data["y"][sel], TPL[c]["crs"])
    wx_sel = data["wx"][sel]                              # copy once per county, not once per file
    hist_cache = {}
    for j, v in enumerate(WXN):
        folder, name = FC_VARS[v]
        files = sorted(glob.glob(os.path.join(FC30_DIR, folder, name, "mean", f"{c}_{name}_mean_*.tif")))
        if not files: print(f"{c} {v}: no forecast files found in {os.path.join(FC30_DIR, folder, name, 'mean')}"); continue
        for f in files:
            mt = re.search(r"_(\d{4})-(\d{2})-(\d{2})\.tif$", f)
            if not mt: continue
            y, m = int(mt.group(1)), int(mt.group(2))
            fc = np.nanmean(sample_tif_nearest(f, pts))
            if v == "rain" and RAIN_UNIT == "mm_month":
                fc *= calendar.monthrange(y, m)[1]
            if (j, m) not in hist_cache:                  # historical yearly means depend only on variable and month
                cols = [mi for mi, (yy, mm) in enumerate(MONTHS) if mm == m]
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    py = np.nanmean(wx_sel[:, cols, j], axis=0)
                hist_cache[(j, m)] = py[np.isfinite(py)]
            per_year = hist_cache[(j, m)]
            if len(per_year) < 2: continue
            mu, sd = per_year.mean(), per_year.std(ddof=1)
            rows.append(dict(county=c, var=v, month=f"{y}-{m:02d}", forecast=fc, hist_mean=mu, hist_min=per_year.min(),
                             hist_max=per_year.max(), z=(fc - mu) / max(sd, 1e-6), ratio=fc / mu if abs(mu) > 1e-6 else np.nan))
cmp_ = pd.DataFrame(rows)
if len(cmp_):
    print(cmp_.round(2).to_string(index=False))
    bad = cmp_[(cmp_.z.abs() > 3) | ((cmp_["var"] == "rain") & ((cmp_.ratio < 0.2) | (cmp_.ratio > 5)))]
    print("\nPossible unit / offset mismatches (|z| > 3, or rain ratio outside 0.2-5):")
    print(bad.round(2).to_string(index=False) if len(bad) else "  none: forecast values sit within the historical range.")
    for v in ("temp", "rh", "wind"):
        d = cmp_[cmp_["var"] == v]
        if len(d): print(f"  mean forecast - historical for {v}: {np.mean(d.forecast - d.hist_mean):+.2f}")
    r = cmp_[cmp_["var"] == "rain"]
    if len(r): print(f"  median forecast / historical ratio for rain: {r.ratio.median():.2f}  (~1 expected; ~0.03 or ~30 would mean daily vs monthly units are mixed again)")

In [ ]:
FC_VARS = {"temp": ("temperature", "temp"), "rain": ("rainfall", "rain"), "rh": ("humidity", "rh"), "wind": ("wind", "wind_speed")}
FLOOR = {"temp": 0.5, "rain": 10.0, "rh": 2.0, "wind": 0.1}   # minimum sd used for z

rows = []
for ci, c in enumerate(COUNTIES):
    sel = np.flatnonzero(cid == ci)[::10]
    pts = Pts(data["x"][sel], data["y"][sel], TPL[c]["crs"])
    wx_sel = data["wx"][sel]
    hist_cache = {}
    for j, v in enumerate(WXN):
        folder, name = FC_VARS[v]
        files = sorted(glob.glob(os.path.join(FC30_DIR, folder, name, "mean", f"{c}_{name}_mean_*.tif")))
        if not files:
            print(f"{c} {v}: no forecast files found"); continue
        for f in files:
            mt = re.search(r"_(\d{4})-(\d{2})-(\d{2})\.tif$", f)
            if not mt: continue
            y, m = int(mt.group(1)), int(mt.group(2))
            fc = np.nanmean(sample_tif_nearest(f, pts))
            if v == "rain" and RAIN_UNIT == "mm_month":
                fc *= calendar.monthrange(y, m)[1]
            if (j, m) not in hist_cache:
                cols = [mi for mi, (yy, mm) in enumerate(MONTHS) if mm == m]
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    py = np.nanmean(wx_sel[:, cols, j], axis=0)
                hist_cache[(j, m)] = py[np.isfinite(py)]
            per_year = hist_cache[(j, m)]
            if len(per_year) < 2: continue
            rows.append(dict(county=c, var=v, month=f"{y}-{m:02d}", forecast=fc,
                             hist_mean=per_year.mean(), hist_sd=per_year.std(ddof=1),
                             hist_min=per_year.min(), hist_max=per_year.max()))
raw = pd.DataFrame(rows)

# one row per county / variable / month: average the forecast dates first
cmp_ = (raw.groupby(["county", "var", "month"], as_index=False)
            .agg(n_dates=("forecast", "size"), forecast=("forecast", "mean"),
                 hist_mean=("hist_mean", "first"), hist_sd=("hist_sd", "first"),
                 hist_min=("hist_min", "first"), hist_max=("hist_max", "first")))
cmp_["sd_used"] = [max(s, FLOOR[v]) for s, v in zip(cmp_.hist_sd, cmp_["var"])]
cmp_["z"] = (cmp_.forecast - cmp_.hist_mean) / cmp_.sd_used
cmp_["ratio"] = np.where(cmp_.hist_mean.abs() > 1e-6, cmp_.forecast / cmp_.hist_mean, np.nan)
cmp_["hist_constant"] = cmp_.hist_min == cmp_.hist_max      # same value every year = broken history

print(cmp_.drop(columns="hist_sd").round(2).to_string(index=False))

bad = cmp_[(cmp_.z.abs() > 3) | cmp_.hist_constant |
           ((cmp_["var"] == "rain") & ((cmp_.ratio < 0.2) | (cmp_.ratio > 5)))]
print("\nFlagged (|z| > 3 with sd floor, constant history, or rain ratio outside 0.2-5):")
print(bad.drop(columns="hist_sd").round(2).to_string(index=False) if len(bad) else "  none")
for v in ("temp", "rh", "wind"):
    d = cmp_[cmp_["var"] == v]
    if len(d): print(f"  mean forecast - historical for {v}: {np.mean(d.forecast - d.hist_mean):+.2f}")
r = cmp_[cmp_["var"] == "rain"]
if len(r): print(f"  median forecast / historical ratio for rain: {r.ratio.median():.2f}")